# 🎮 AI Game Dev — GPU Reinforcement Learning Training

Trains the AI to produce **better games** through trial, error, rewards and penalties — on a **free Colab GPU** (T4/A100).

### How it works
| Phase | What happens |
|-------|--------------|
| 1 | AI generates pixel art, GDScript, game designs via PixelLab + LLM |
| 2 | Evaluator scores every output (0–100) on Art / Code / Design axes |
| 3 | PPO policy gradient updates agent weights toward higher scores |
| 4 | ONNX checkpoint exported, pushed back to GitHub |

### Curriculum levels
| Level | Min Score | Tasks |
|-------|-----------|-------|
| 1 Basics | 0+ | Simple characters, tiles, 20-line scripts |
| 2 Intermediate | 40+ | Animated chars, full scenes, enemy AI |
| 3 Advanced | 65+ | Complete game scenes, systems |
| 4 Expert | 80+ | Full polished isekai RPG |

---
**Before running**: set your secrets in the cell below (or use Colab Secrets via the 🔑 icon).

In [6]:
# ── AUTO-RECONNECT: keeps Colab alive (run this cell FIRST) ──────────────────
# Runs a JS snippet in the browser that clicks 'Reconnect' every 60s.
# This prevents Colab from disconnecting while training.
from IPython.display import display, Javascript
display(Javascript('''
  function keepAlive() {
    var reconnect = document.querySelector("colab-toolbar-button#connect");
    if (reconnect && reconnect.shadowRoot) {
      var btn = reconnect.shadowRoot.querySelector("paper-icon-button");
      if (btn) btn.click();
    }
    // Also click run-anyway if a dialog appears
    document.querySelectorAll("paper-button").forEach(btn => {
      if (btn.innerText === 'RUN ANYWAY') btn.click();
    });
  }
  setInterval(keepAlive, 60000);
  console.log('Auto-reconnect enabled — Colab will stay connected.');
'''))
print('✅ Auto-reconnect enabled — Colab will stay connected during training')
print('   Leave this tab open. Training can run for 12h before needing manual reconnect.')


<IPython.core.display.Javascript object>

✅ Auto-reconnect enabled — Colab will stay connected during training
   Leave this tab open. Training can run for 12h before needing manual reconnect.


In [7]:
# ── 0. GPU check ──────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU detected:', result.stdout.strip())
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → GPU (T4 is free)')

import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

✅ GPU detected: Tesla T4, 15360 MiB
PyTorch 2.10.0+cu128 | CUDA available: True
Using device: cuda


In [8]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q pillow numpy matplotlib requests aiohttp groq anthropic google-generativeai
!pip install -q onnx onnxruntime
# torch is pre-installed on Colab — skip re-install to save time
print('✅ Dependencies ready')

✅ Dependencies ready


In [9]:
# Clone / pull repo
import os
from pathlib import Path

GITHUB_USER  = 'odanebillit12-png'
GITHUB_REPO  = 'rl-training-colab'
REPO_DIR = Path(f'/content/{GITHUB_REPO}')

# Load API keys from Colab Secrets if available
GROQ_KEY = GEMINI_KEY = ANTHROPIC_KEY = PIXELLAB_KEY = ''
try:
    from google.colab import userdata  # type: ignore
    GROQ_KEY      = userdata.get('GROQ_API_KEY') or ''
    GEMINI_KEY    = userdata.get('GEMINI_API_KEY') or ''
    ANTHROPIC_KEY = userdata.get('ANTHROPIC_API_KEY') or ''
    PIXELLAB_KEY  = userdata.get('PIXELLAB_API_KEY') or ''
    print('✅ Loaded secrets from Colab Secrets')
except Exception:
    print('ℹ️  No Colab Secrets found')

os.environ['GROQ_API_KEY']      = GROQ_KEY
os.environ['GEMINI_API_KEY']    = GEMINI_KEY
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_KEY
os.environ['PIXELLAB_API_KEY']  = PIXELLAB_KEY

REPO_URL = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'

if REPO_DIR.exists():
    print('📦 Pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull --quiet')
else:
    print('📦 Cloning repo...')
    os.system(f'git clone --quiet {REPO_URL} {REPO_DIR}')

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, str(REPO_DIR))
print(f'✅ Repo ready at {REPO_DIR}')


ℹ️  No Colab Secrets found
📦 Cloning repo...
✅ Repo ready at /content/rl-training-colab


In [10]:
# ── 3. PPO Policy — GPU-accelerated ──────────────────────────────────────────
import torch  # type: ignore
import torch.nn as nn  # type: ignore
import torch.optim as optim  # type: ignore
import numpy as np
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
import json
from pathlib import Path

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── State encoder (converts score dict → feature vector) ─────────────────────
STATE_DIM  = 16   # art_score, code_score, design_score, level, episode, ...
ACTION_DIM = 8    # one action type per slot
ACTION_NAMES = [
    'draw_character', 'draw_tile', 'draw_prop',
    'generate_scene_code', 'generate_player_code', 'generate_enemy_code',
    'design_map', 'design_game',
]

class PolicyNet(nn.Module):
    """Actor-Critic network for PPO."""
    def __init__(self, state_dim=STATE_DIM, action_dim=ACTION_DIM, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
        )
        self.actor  = nn.Linear(hidden, action_dim)   # policy logits
        self.critic = nn.Linear(hidden, 1)            # value estimate

    def forward(self, x):
        h = self.shared(x)
        return self.actor(h), self.critic(h)

    def act(self, state: torch.Tensor):
        logits, value = self(state)
        dist   = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        return action, dist.log_prob(action), value

@dataclass
class Trajectory:
    states:      List[torch.Tensor] = field(default_factory=list)
    actions:     List[torch.Tensor] = field(default_factory=list)
    log_probs:   List[torch.Tensor] = field(default_factory=list)
    values:      List[torch.Tensor] = field(default_factory=list)
    rewards:     List[float]        = field(default_factory=list)
    dones:       List[bool]         = field(default_factory=list)

class PPOTrainer:
    """Proximal Policy Optimisation trainer (GPU-accelerated)."""

    def __init__(self, lr=3e-4, gamma=0.99, lam=0.95, clip=0.2,
                 entropy_coef=0.01, value_coef=0.5, epochs=4):
        self.policy  = PolicyNet().to(device)
        self.optim   = optim.Adam(self.policy.parameters(), lr=lr)
        self.gamma   = gamma
        self.lam     = lam
        self.clip    = clip
        self.ent_c   = entropy_coef
        self.val_c   = value_coef
        self.epochs  = epochs
        self.traj    = Trajectory()
        self.step    = 0

        # Try to load existing checkpoint
        ckpt = Path('training_data/ppo_policy.pt')
        if ckpt.exists():
            self.policy.load_state_dict(torch.load(ckpt, map_location=device))
            print(f'✅ Loaded checkpoint from {ckpt}')
        else:
            print('🆕 Starting fresh PPO policy')

    def encode_state(self, scores: Dict, episode: int, level: int) -> torch.Tensor:
        """Convert score dict → normalised feature vector."""
        art    = scores.get('art_score',    0) / 100.0
        code   = scores.get('code_score',   0) / 100.0
        design = scores.get('design_score', 0) / 100.0
        total  = scores.get('total_score',  0) / 100.0
        ep_n   = min(episode / 200.0, 1.0)
        lvl_n  = level / 4.0
        # pad to STATE_DIM
        vec = [art, code, design, total, ep_n, lvl_n] + [0.0] * (STATE_DIM - 6)
        return torch.tensor(vec, dtype=torch.float32, device=device)

    def compute_gae(self) -> Tuple[torch.Tensor, torch.Tensor]:
        """Generalised Advantage Estimation."""
        rewards  = self.traj.rewards
        values   = [v.item() for v in self.traj.values]
        dones    = self.traj.dones
        T        = len(rewards)
        advs     = [0.0] * T
        last_gae = 0.0
        last_val = 0.0

        for t in reversed(range(T)):
            next_val  = last_val if not dones[t] else 0.0
            delta     = rewards[t] + self.gamma * next_val - values[t]
            last_gae  = delta + self.gamma * self.lam * last_gae * (not dones[t])
            advs[t]   = last_gae
            last_val  = values[t]

        advs    = torch.tensor(advs, dtype=torch.float32, device=device)
        returns = advs + torch.tensor(values, dtype=torch.float32, device=device)
        advs    = (advs - advs.mean()) / (advs.std() + 1e-8)
        return advs, returns

    def update(self):
        """Run PPO update on collected trajectory."""
        if len(self.traj.rewards) < 4:
            return {}

        advs, returns = self.compute_gae()
        states    = torch.stack(self.traj.states)
        actions   = torch.stack(self.traj.actions)
        old_lps   = torch.stack(self.traj.log_probs).detach()

        total_loss = total_pg = total_vl = total_ent = 0.0
        for _ in range(self.epochs):
            logits, values = self.policy(states)
            dist    = torch.distributions.Categorical(logits=logits)
            new_lps = dist.log_prob(actions)
            entropy = dist.entropy().mean()

            ratio   = torch.exp(new_lps - old_lps)
            pg_loss = -torch.min(
                ratio * advs,
                ratio.clamp(1 - self.clip, 1 + self.clip) * advs
            ).mean()
            val_loss = 0.5 * (values.squeeze() - returns).pow(2).mean()
            loss     = pg_loss + self.val_c * val_loss - self.ent_c * entropy

            self.optim.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            self.optim.step()

            total_loss += loss.item()
            total_pg   += pg_loss.item()
            total_vl   += val_loss.item()
            total_ent  += entropy.item()

        # Clear trajectory
        self.traj = Trajectory()
        n = self.epochs
        return {'loss': total_loss/n, 'pg': total_pg/n, 'vl': total_vl/n, 'ent': total_ent/n}

    def save_checkpoint(self, path='training_data/ppo_policy.pt'):
        Path(path).parent.mkdir(exist_ok=True)
        torch.save(self.policy.state_dict(), path)

    def export_onnx(self, path='training_data/ppo_policy.onnx'):
        Path(path).parent.mkdir(exist_ok=True)
        dummy = torch.zeros(1, STATE_DIM, device=device)
        torch.onnx.export(
            self.policy, dummy, path,
            input_names=['state'], output_names=['logits', 'value'],
            dynamic_axes={'state': {0: 'batch'}},
            opset_version=11
        )
        print(f'✅ ONNX exported to {path}')

ppo = PPOTrainer()
print(f'Policy parameters: {sum(p.numel() for p in ppo.policy.parameters()):,}')

🆕 Starting fresh PPO policy
Policy parameters: 19,849


In [12]:
# ── 4. Connect AI Dev trainers ─────────────────────────────────────────────────
import asyncio
from ai_game_agent.training.game_evaluator   import GameEvaluator
from ai_game_agent.training.experience_memory import ExperienceMemory
from ai_game_agent.training.drawing_trainer  import DrawingTrainer, DrawingConfig
from ai_game_agent.training.rl_trainer       import RLTrainer, TrainingConfig

evaluator = GameEvaluator(groq_api_key=GROQ_KEY, gemini_api_key=GEMINI_KEY)
memory    = ExperienceMemory()

# Load existing memory if available
mem_file = Path('training_data/experience_memory.json')
if mem_file.exists():
    memory.load(str(mem_file))
    print(f'✅ Loaded {len(memory.episodes)} prior episodes from memory')

rl = RLTrainer(TrainingConfig(max_episodes=1, target_score=99))
print('✅ Trainers ready')

TypeError: GameEvaluator.__init__() got an unexpected keyword argument 'groq_api_key'

In [ ]:
# -- 5. Self-restarting training loop (runs forever, resumes on reconnect) ----
# To STOP: run the Stop cell below, or Interrupt kernel (Runtime -> Interrupt).
# On reconnect, re-run from Cell 3 onward -- it will resume from last checkpoint.
# -----------------------------------------------------------------------------
import asyncio, json, time
from ai_game_agent.training.rl_trainer import RLTrainer, TrainingConfig
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import clear_output

DATA_DIR      = Path('training_data')
DATA_DIR.mkdir(exist_ok=True)
SESSION_FILE  = DATA_DIR / 'colab_session.json'
STOP_FLAG     = DATA_DIR / 'stop_colab.flag'
CKPT_PT       = DATA_DIR / 'ppo_policy.pt'
CKPT_ONNX     = DATA_DIR / 'ppo_policy.onnx'

EPISODES_PER_BATCH = 50
SAVE_EVERY         = 10
PUSH_EVERY         = 50
UPDATE_EVERY       = 10
CHART_EVERY        = 5

CURRICULUM = [
    {'level': 1, 'min_score': 0,  'name': 'Basics',       'action': 'draw_character'},
    {'level': 2, 'min_score': 40, 'name': 'Intermediate', 'action': 'generate_scene_code'},
    {'level': 3, 'min_score': 65, 'name': 'Advanced',     'action': 'draw_character'},
    {'level': 4, 'min_score': 80, 'name': 'Expert',       'action': 'design_game'},
]

def load_session():
    if SESSION_FILE.exists():
        s = json.loads(SESSION_FILE.read_text())
        print(f'Resuming from ep {s["total_episodes"]} | avg={s["last_avg10"]:.1f} | lvl={s["curriculum_lvl"]}')
        return s
    return {'total_episodes': 0, 'last_avg10': 0, 'curriculum_lvl': 1,
            'scores_hist': [], 'reward_hist': [], 'loss_hist': []}

def save_session(ep_total, avg10, lvl, scores, rewards, losses):
    SESSION_FILE.write_text(json.dumps({
        'total_episodes': ep_total,
        'last_avg10':     float(avg10),
        'curriculum_lvl': lvl,
        'scores_hist':    [float(x) for x in scores[-500:]],
        'reward_hist':    [float(x) for x in rewards[-500:]],
        'loss_hist':      [float(x) for x in losses[-200:]],
        'saved_at':       time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }, indent=2))

def push_to_github(ep_total):
    if not GH_PAT:
        return
    import subprocess
    subprocess.run(['git', 'add', str(CKPT_PT), str(CKPT_ONNX), str(SESSION_FILE)], capture_output=True)
    r = subprocess.run(['git', 'commit', '-m', f'Colab GPU checkpoint ep={ep_total}'],
                       capture_output=True, text=True)
    if 'nothing to commit' not in r.stdout:
        subprocess.run(['git', 'push'], capture_output=True)
        print(f'  Pushed checkpoint at ep={ep_total}')

def get_reward(score, prev_avg):
    return max(0.0, (score - prev_avg) / 100.0) * 2.0 + (score / 100.0) * 0.5 + (-0.5 if score < 30 else 0.0)

async def run_episode(ep, level):
    captured: dict = {}
    def _on_ep(data: dict) -> None:
        captured.update(data)
    try:
        one = RLTrainer(TrainingConfig(max_episodes=1, target_score=99), on_episode=_on_ep)
        await asyncio.wait_for(one.run(), timeout=120)
        s = float(captured.get('score', 50))
        scores: dict = {'art_score': s, 'code_score': s, 'design_score': s}
    except Exception as e:
        scores = {'art_score': 30.0, 'code_score': 30.0, 'design_score': 30.0, 'error': str(e)[:80]}
    scores['total_score'] = float(scores.get('art_score',0)*0.4 + scores.get('code_score',0)*0.3 + scores.get('design_score',0)*0.3)
    return scores

def draw_chart(scores, rewards, losses, ep_total, lvl, elapsed):
    clear_output(wait=True)
    avg10 = float(np.mean(scores[-10:])) if scores else 0
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    axes[0].plot(scores, 'b-', alpha=0.5)
    if len(scores) >= 10:
        ma = np.convolve(scores, np.ones(10)/10, mode='valid')
        axes[0].plot(range(9, len(scores)), ma, 'r-', lw=2, label='MA-10')
    for thr, col, lbl in [(40,'orange','Lvl2'),(65,'green','Lvl3'),(80,'purple','DIVINE')]:
        axes[0].axhline(thr, ls='--', c=col, alpha=0.7, label=lbl)
    axes[0].set_title(f'Score ep={ep_total} | lvl={lvl} | avg={avg10:.1f}')
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 100)
    axes[1].plot(rewards, 'g-', alpha=0.6)
    axes[1].axhline(0, ls='--', c='grey')
    axes[1].set_title('PPO Reward')
    if losses:
        axes[2].plot(losses, 'r-')
        axes[2].set_title('Policy Loss')
    plt.suptitle(f'AI Dev Continuous GPU Training | {elapsed:.1f}min | {device}', fontsize=12)
    plt.tight_layout(); plt.show()

# -- Main infinite loop -------------------------------------------------------
sess           = load_session()
ep_total       = sess['total_episodes']
curriculum_lvl = sess['curriculum_lvl']
scores_hist    = sess['scores_hist']
reward_hist    = sess['reward_hist']
loss_hist      = sess['loss_hist']
avg10          = sess['last_avg10']
run_start      = time.time()
elapsed        = 0.0

if STOP_FLAG.exists():
    STOP_FLAG.unlink()
    print('Cleared previous stop flag')

print(f'Continuous training started on {device.upper()}')
print(f'Resuming from global episode {ep_total}')
print('To stop: run the Stop cell, or use Runtime -> Interrupt')

try:
    while True:
        for batch_ep in range(EPISODES_PER_BATCH):
            if STOP_FLAG.exists():
                print('Stop flag detected -- saving and exiting')
                raise StopIteration

            ep_total += 1
            elapsed   = (time.time() - run_start) / 60

            avg10 = float(np.mean(scores_hist[-10:])) if scores_hist else 0
            for c in CURRICULUM:
                if avg10 >= c['min_score']:
                    curriculum_lvl = c['level']

            state = ppo.encode_state({'art_score': avg10, 'total_score': avg10}, ep_total, curriculum_lvl)
            action, log_prob, value = ppo.policy.act(state)

            scores = asyncio.get_event_loop().run_until_complete(run_episode(ep_total, curriculum_lvl))
            total  = scores['total_score']
            reward = get_reward(float(total), float(avg10))

            ppo.traj.states.append(state)
            ppo.traj.actions.append(action)
            ppo.traj.log_probs.append(log_prob)
            ppo.traj.values.append(value.squeeze())
            ppo.traj.rewards.append(reward)
            ppo.traj.dones.append(False)

            scores_hist.append(total)
            reward_hist.append(reward)

            if ep_total % UPDATE_EVERY == 0:
                metrics = ppo.update()
                if metrics:
                    loss_hist.append(metrics.get('loss', 0))

            if ep_total % SAVE_EVERY == 0:
                ppo.save_checkpoint(str(CKPT_PT))
                save_session(ep_total, avg10, curriculum_lvl, scores_hist, reward_hist, loss_hist)

            if ep_total % PUSH_EVERY == 0:
                ppo.export_onnx(str(CKPT_ONNX))
                push_to_github(ep_total)

            if ep_total % CHART_EVERY == 0:
                draw_chart(scores_hist, reward_hist, loss_hist, ep_total, curriculum_lvl, elapsed)
                print(f'ep={ep_total:5d} | score={total:.1f} | reward={reward:+.3f} | lvl={curriculum_lvl} | {elapsed:.1f}min')

        ppo.save_checkpoint(str(CKPT_PT))
        save_session(ep_total, avg10, curriculum_lvl, scores_hist, reward_hist, loss_hist)
        print(f'Batch done -- ep={ep_total} | avg10={avg10:.1f} | level={curriculum_lvl} | {elapsed:.1f}min')

except (StopIteration, KeyboardInterrupt):
    pass

ppo.save_checkpoint(str(CKPT_PT))
ppo.export_onnx(str(CKPT_ONNX))
save_session(ep_total, avg10, curriculum_lvl, scores_hist, reward_hist, loss_hist)
push_to_github(ep_total)
draw_chart(scores_hist, reward_hist, loss_hist, ep_total, curriculum_lvl, elapsed)
final_avg = float(np.mean(scores_hist[-10:])) if scores_hist else 0
print(f'Saved at ep={ep_total} | avg10={final_avg:.1f}')
print('Re-run this cell to continue from where you left off.')


In [ ]:
# -- 6. Analysis chart (run any time to inspect training progress) -----------
import json, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

sf = Path('training_data/colab_session.json')
if not sf.exists():
    print('No session data yet -- run the training loop first.')
else:
    s       = json.loads(sf.read_text())
    scores  = s['scores_hist']
    rewards = s['reward_hist']
    losses  = s['loss_hist']
    ep_total= s['total_episodes']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    axes[0,0].plot(scores, alpha=0.5)
    if len(scores) >= 10:
        ma = np.convolve(scores, np.ones(10)/10, mode='valid')
        axes[0,0].plot(range(9, len(scores)), ma, 'r-', lw=2, label='MA-10')
    for thr, col, lbl in [(40,'orange','Lvl2'),(65,'green','Lvl3'),(80,'purple','DIVINE')]:
        axes[0,0].axhline(thr, ls='--', c=col, alpha=0.7, label=lbl)
    axes[0,0].set_title(f'Score trajectory (ep={ep_total})')
    axes[0,0].legend(); axes[0,0].set_ylim(0,100)
    axes[0,1].plot(rewards, 'g-', alpha=0.6)
    axes[0,1].axhline(0, ls='--', c='grey')
    axes[0,1].set_title('PPO reward signal')
    axes[1,0].hist(scores, bins=20, color='steelblue', edgecolor='white')
    axes[1,0].axvline(np.mean(scores), c='red', label=f'Mean={np.mean(scores):.1f}')
    axes[1,0].set_title('Score distribution'); axes[1,0].legend()
    if losses:
        axes[1,1].plot(losses, 'r-'); axes[1,1].set_title('PPO policy loss')
    plt.suptitle(f'AI Dev -- ep={ep_total} | avg10={s["last_avg10"]:.1f} | level={s["curriculum_lvl"]}', fontsize=13)
    plt.tight_layout()
    plt.savefig('training_data/colab_training_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_data/colab_training_summary.png')


In [ ]:
# -- 7. Stop training gracefully ----------------------------------------------
# Run this cell to stop the training loop (it saves before exiting).
from pathlib import Path
Path('training_data/stop_colab.flag').touch()
print('Stop flag created -- training will save and exit after the current episode.')
